# 🎌 mangavid en Google Colab: historias de anime narradas

1. **Activa la GPU:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU* (o mejor, si tienes Colab Pro).
2. Ejecuta las celdas en orden. Las tareas largas (descargar modelos, generar imágenes y vídeo) van **en segundo plano**, con una celda que enseña el progreso cada 30 s.
3. **Para que no se apague la sesión:** usa el script del paso 4 en la consola del navegador.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive y código

Las historias y los vídeos se guardan en **Google Drive** (`Mi unidad/mangavid`) para no perderlos si se corta la sesión.
Los modelos (~25 GB) se descargan en el disco de Colab: no caben en los 15 GB gratis de Drive y en Colab se bajan en pocos minutos.

In [ ]:
#@title Conectar Drive, descargar el código e instalar
import os, getpass
from google.colab import drive
drive.mount("/content/drive")
CARPETA = "/content/drive/MyDrive/mangavid"  #@param {type:"string"}
os.makedirs(CARPETA, exist_ok=True)

REPO = "/content/doblaje"
if not os.path.exists(REPO):
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    if not token:
        token = getpass.getpass("Token de GitHub (el repositorio es privado): ")
    !git clone -q https://{token}@github.com/statsuya500/yousef-research-engine.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt
print("✅ Listo. Tus historias van en", CARPETA)

In [ ]:
#@title Utilidades: lanzar tareas largas en segundo plano y vigilarlas
import subprocess, time, shutil, datetime
from IPython.display import clear_output

def lanzar(orden, log, sin_conexion=True):
    """Ejecuta una orden en segundo plano y guarda lo que escribe en `log`."""
    env = dict(os.environ)
    if sin_conexion:
        env["HF_HUB_OFFLINE"] = "1"
    else:
        env.pop("HF_HUB_OFFLINE", None)
    salida = open(log, "w")
    proceso = subprocess.Popen(orden, cwd=REPO, env=env, stdout=salida, stderr=subprocess.STDOUT, text=True)
    print(f"▶ En marcha (proceso {proceso.pid}). Log: {log}")
    return proceso

def vigilar(proceso, log, cada=30, lineas=12):
    """Enseña el final del log cada `cada` segundos hasta que termine.
    Si paras esta celda, la tarea SIGUE: vuelve a ejecutar vigilar() para verla."""
    inicio = time.time()
    while proceso.poll() is None:
        clear_output(wait=True)
        libre = shutil.disk_usage(REPO).free / 1e9
        minutos = (time.time() - inicio) / 60
        print(f"⏳ Sigue activo · {datetime.datetime.now():%H:%M:%S} · {minutos:.0f} min · disco libre {libre:.0f} GB")
        print("─" * 60)
        try:
            print("".join(open(log, encoding="utf-8", errors="replace").readlines()[-lineas:]))
        except FileNotFoundError:
            pass
        time.sleep(cada)
    clear_output(wait=True)
    print("".join(open(log, encoding="utf-8", errors="replace").readlines()[-lineas:]))
    print("✅ Terminado" if proceso.returncode == 0 else f"❌ Terminó con error {proceso.returncode}: mira el log")

## 3. Descargar los modelos (unos 5-10 minutos en Colab)

In [ ]:
#@title Descargar todos los modelos en segundo plano
SIN_WAI = False  #@param {type:"boolean"}
orden = ["python", "-m", "mangavid", "modelos"] + (["--sin-wai"] if SIN_WAI else [])
descarga = lanzar(orden, "/content/modelos.log", sin_conexion=False)
vigilar(descarga, "/content/modelos.log")

## 4. Que no se apague la sesión

Colab desconecta la sesión si pasa un rato sin que toques la página. Para evitarlo mientras descarga o genera:

1. En esta pestaña de Colab, pulsa **F12** (o *clic derecho → Inspeccionar*) y abre la pestaña **Console / Consola**.
2. Pega este script y pulsa **Enter**. Si Chrome no te deja pegar, escribe primero `allow pasting` y pulsa Enter.

```javascript
function mantenerColab() {
  const boton = document.querySelector("colab-connect-button");
  const conectar = boton && boton.shadowRoot && boton.shadowRoot.querySelector("#connect");
  if (conectar) conectar.click();
  else document.querySelector("#top-toolbar")?.click();
  console.log("Colab sigue activo:", new Date().toLocaleTimeString());
}
const vigilante = setInterval(mantenerColab, 60 * 1000);
// Para pararlo: clearInterval(vigilante)
```

Cada minuto pulsa el botón de conexión de Colab y escribe la hora en la consola. Déjalo así y **no cierres la pestaña**.
El script se pierde si recargas la página: entonces vuelve a pegarlo.

Límites de Colab que no se pueden saltar: la sesión gratuita dura como mucho unas 12 horas, y a veces Colab
pregunta si sigues ahí con una casilla que hay que marcar a mano. Por eso todo se guarda en Drive y por partes:
si se corta, al relanzar solo se rehace lo que falta.

## 5. Hacer una historia

Sube tu `historia.txt` a `Mi unidad/mangavid/<nombre>/` y pon aquí su ruta. Cada paso va en segundo plano con su vigilante.

In [ ]:
#@title Elegir historia y modelo
HISTORIA = CARPETA + "/mi_historia/historia.txt"  #@param {type:"string"}
MODELO = "John6666/wai-nsfw-illustrious-sdxl-v150-sdxl"  #@param ["John6666/wai-nsfw-illustrious-sdxl-v150-sdxl", "cagliostrolab/animagine-xl-4.0"]
assert os.path.exists(HISTORIA), f"No encuentro {HISTORIA}"
print("Historia:", HISTORIA, "· modelo:", MODELO)

In [ ]:
#@title Paso 1: retratos de los personajes
p = lanzar(["python", "-m", "mangavid", "personajes", HISTORIA, "--model", MODELO], "/content/personajes.log")
vigilar(p, "/content/personajes.log")

In [ ]:
#@title Ver las opciones de retrato y elegir
import glob
from IPython.display import Image, display
carpeta_fichas = HISTORIA.rsplit(".", 1)[0] + "_personajes"
for f in sorted(glob.glob(carpeta_fichas + "/*_[0-9].png")):
    print(os.path.basename(f)); display(Image(f, width=220))
# Para elegir otra opción:  !python -m mangavid elegir "{HISTORIA}" <personaje> <número>

In [ ]:
#@title Paso 2: imágenes de las escenas (lo más largo)
p = lanzar(["python", "-m", "mangavid", "run", HISTORIA, "--model", MODELO, "--only-images"], "/content/escenas.log")
vigilar(p, "/content/escenas.log")

In [ ]:
#@title Paso 3: voz, movimiento, subtítulos y vídeo
VOZ = "es-MX-JorgeNeural"  #@param ["es-MX-JorgeNeural", "es-ES-AlvaroNeural", "es-ES-ElviraNeural", "es-MX-DaliaNeural"]
p = lanzar(["python", "-m", "mangavid", "run", HISTORIA, "--model", MODELO, "--voice", VOZ], "/content/video.log")
vigilar(p, "/content/video.log")
print("Vídeo:", HISTORIA.rsplit(".", 1)[0] + ".mp4", "(en tu Google Drive)")

In [ ]:
#@title Paso 4: miniatura
p = lanzar(["python", "-m", "mangavid", "miniatura", HISTORIA, "--model", MODELO], "/content/miniatura.log")
vigilar(p, "/content/miniatura.log")
display(Image(HISTORIA.rsplit(".", 1)[0] + "_miniaturas/opciones.jpg", width=900))